# Module 1: Content Addressable Storage

Welcome to the first module of your journey in building a Version Control System (VCS). 

Most of the time, we think of files in terms of their **name** and **location** (e.g., `src/main.py`). This is called *location-addressable storage*. If you change the content of `main.py`, it's still the same file at the same location.

Git does things differently. It uses **Content Addressable Storage**. In this system, the "address" of a piece of data is derived from the **content itself**. If the content changes, the address changes. This is the fundamental secret to how Git handles snapshots, deduplication, and integrity.

## 1. The Magic of Hashing

To turn content into an address, we need a way to generate a unique, fixed-length string (a fingerprint) for any amount of data. This is called **Hashing**.

Git uses the **SHA-1** (Secure Hash Algorithm 1). A SHA-1 hash is always 40 hexadecimal characters long, regardless of whether you are hashing a single letter or a 10GB movie.

### Properties of a good hash for VCS:
1. **Deterministic**: The same input always produces the same hash.
2. **Fast**: It should be quick to compute.
3. **Collision Resistant**: It should be extremely rare for two different inputs to produce the same hash.
4. **Avalanche Effect**: A tiny change in the input (like changing a comma to a period) should result in a completely different hash.

In [7]:
import hashlib

def get_sha1_hash(content: bytes) -> str:
    """Returns the SHA-1 hash of the given byte content."""
    return hashlib.sha1(content).hexdigest()

# Testing the avalanche effect
text1 = b"Hello World"
text2 = b"Hello world"

print(text1.decode("utf-8"))
print(f"Hash 1: {get_sha1_hash(text1)}")
hash_2 = get_sha1_hash(text2)
print(f"Hash 2: {hash_2}")
print(f"Hash length: {len(hash_2)}")
print(f"Changed only the 'W' to 'w' and the hashes are completely different!")

Hello World
Hash 1: 0a4d55a8d778e5022fab701977c5d840bbc486d0
Hash 2: 7b502c3a1f48c8609ae212cdfb639dee39673f5e
Hash length: 40
Changed only the 'W' to 'w' and the hashes are completely different!


## 2. What is a "Blob"?

In Git, the word **blob** stands for "Binary Large Object". 

A blob is the simplest object in Git. It stores the **content** of a file, but it does **not** store the filename, the permissions, or the creation date. That information is stored elsewhere (in "Trees"), which we will learn in Module 2.

### How Git stores blobs:
1. Take the file content.
2. Calculate the SHA-1 hash.
3. Save the content into a file named after that hash in the `.git/objects` directory.

**Immutability**: Once a blob is written, it never changes. If you edit your file, Git doesn't modify the old blob; it creates a *new* blob with a *new* hash. This is why you can never "lose" a version of a file in Git—as long as the blob exists, the data is safe.

In [15]:
import os
import hashlib

class SimpleVCS:
    def __init__(self, root_dir=".pygit"):
        self.root_dir = root_dir
        self.objects_dir = os.path.join(self.root_dir, "objects")
        
        # Ensure the objects directory exists
        os.makedirs(self.objects_dir, exist_ok=True)

    def store_blob(self, content: str) -> str:
        """
        Stores content as a blob and returns its hash (address).
        """
        # Convert string to bytes
        data = content.encode('utf-8')
        
        # 1. Calculate hash
        obj_hash = hashlib.sha1(data).hexdigest()
        
        # 2. Define path: .pygit/objects/<hash>
        path = os.path.join(self.objects_dir, obj_hash)
        
        # 3. Store if it doesn't already exist (deduplication!)
        if not os.path.exists(path):
            with open(path, "wb") as f:
                f.write(data)
        
        return obj_hash

    def retrieve_blob(self, obj_hash: str) -> str:
        """
        Retrieves the content of a blob given its hash.
        """
        path = os.path.join(self.objects_dir, obj_hash)
        if not os.path.exists(path):
            raise FileNotFoundError(f"Object {obj_hash} not found!")
            
        with open(path, "rb") as f:
            return f.read().decode('utf-8')

# Let's try it out
vcs = SimpleVCS()
hash1 = vcs.store_blob("Hello, this is my first file!")
hash2 = vcs.store_blob("Hello, this is my first file!") # Same content
hash3 = vcs.store_blob("Hello, this is a different file!")
hash4 = "Hello!"

print(f"File 1 Hash: {hash1}")
print(f"File 2 Hash: {hash2}")
print(f"File 3 Hash: {hash3}")
print(f"Are File 1 and 2 the same object? {hash1 == hash2}") # Should be True
print(f"Content of hash1: {vcs.retrieve_blob(hash1)}")
# print(f"Content of a non existing hash: {vcs.retrieve_blob(hash4)}")

File 1 Hash: b58004db38afe018a4e9d6bd301111333b3b9292
File 2 Hash: b58004db38afe018a4e9d6bd301111333b3b9292
File 3 Hash: 33f4cf12ec0443ebb1f649c35a4421b8f6245353
Are File 1 and 2 the same object? True
Content of hash1: Hello, this is my first file!


## 🎓 Summary & Review

In this session, we learned that Git doesn't track files by name, but by their **content**.
- **SHA-1** turns any content into a unique 40-character address.
- A **Blob** is just the content of a file stored under its hash.
- **Deduplication**: If two files have the exact same content, Git only stores one blob, even if the files have different names in different folders.

### Quick Questions:
1. If I have a file `A.txt` and I copy it to `B.txt` without changing any text, how many blobs will be created in the `.pygit/objects` folder?
2. Why is it important that the hash is "deterministic"?
3. What happens to the old blob when you edit a file and store it again?

### Answer to the questions:
1. Since git keeps track of the content and the filename is irrelevant, there will be no new blob created in the objects folder as the contents of the both the files are
exactly same, evne though the names of the files are different
2. Same input should always produce same hash, as it is important to store the version of the file, if there is new hash generated in each iteration it is treated as 
totally different, that is why it is important to have a hash that is deterministic
3. The old bolb continues to exist in the objects folder along with the new blob that is generated after the contents of the file have been edited. This is the core
essence of having a Version Control System as it holds every version of the file

## 🛠️ Code Challenge: The File-to-Blob Pipeline

Now it's your turn. You have the `SimpleVCS` class, but it currently only handles strings in memory.

**Your Task:**
Implement a new method for the `SimpleVCS` class called `hash_file(self, file_path: str) -> str`.

This method should:
1. Read the content of a file from the disk using the provided `file_path`.
2. Use the existing `store_blob` method to save that content into the `.pygit` storage.
3. Return the resulting hash.

**How to submit:**
Write your implementation in a new code cell below. Once you're done, tell me you've finished and show me your code. I will review it, ask you clarifying questions, and help you refine it before we move to Module 2!

In [17]:
# Solution to the coding challenge

import os
import hashlib

class VCS_v1:
    def __init__(self, root_dir='.pygit'):
        self.root_dir = root_dir
        self.objects_dir = os.path.join(self.root_dir, 'objects')

        os.makedirs(self.objects_dir, exist_ok=True)

    def store_blob(self, content: bytes) -> str:
        object_hash = hashlib.sha1(content).hexdigest()
        path = os.path.join(self.objects_dir, object_hash)

        if not os.path.exists(path):
            with open(path, "wb") as f:
                f.write(content)
        return object_hash

    def retrieve_blob(self, object_hash: str) -> bytes:
        path = os.path.join(self.objects_dir, object_hash)
        if not os.path.exists(path):
            raise FileNotFoundError(f"Object {object_hash} not found!")

        with open(path, "rb") as f:
            return f.read()

    def hash_file(self, file_path: str) -> str:
        """
        Reads the contents of the file provided from the given path
        (if the file exists) and returns the hash
        """

        # Check if the file exists
        if not os.path.exists(file_path):
            raise FileNotFoundError(f"The described file {file_path} does not exists.")

        with open(file_path, 'rb') as f:
            file_content = f.read()

        return self.store_blob(file_content)
vcs_1 = VCS_v1()
hash_test_txt = vcs_1.hash_file('./test.txt')
print(f"Hash of test.txt file: {hash_test_txt}")

print(f"Contents of the test.txt file: {vcs_1.retrieve_blob(hash_test_txt)}")
            

Hash of test.txt file: 4bcf82e72c6a1bb1d8b3c2249dfec232f1d6de3d
Contents of the test.txt file: Hello, this is a simple test file
